# Chain Model Training - US Traffic Incident Analysis
ในโปรเจกต์นี้เราจะใช้แนวทาง **Chain Model**:
1. **Model 1 (Distance Predictor):** ทำนายระยะทางของอุบัติเหตุ (`Distance(mi)`)
2. **Model 2 (Duration Predictor):** ทำนายระยะเวลาของอุบัติเหตุ (`Duration(min)`) โดยใช้ `Distance(mi)` เป็นหนึ่งใน Feature

**ทำไมถึงต้องทำแบบนี้?** เพราะในสถานการณ์จริง (หรือใน Test Set) เราจะไม่ทราบระยะทางล่วงหน้า เราจึงต้องทำนายระยะทางก่อน เพื่อนำค่านั้นมาช่วยทำนายระยะเวลาที่แม่นยำขึ้น

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Sklearn & Pipeline
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# Model
from xgboost import XGBRegressor

## 1. Load Data

In [2]:
df_train = pd.read_csv("../data/processed/03/train_features.csv")
print(f"Train set shape: {df_train.shape}")

MemoryError: Unable to allocate 1.22 GiB for an array with shape (30, 5469092) and data type float64

## 2. Configuration & Features
กำหนดคอลัมน์ที่จะใช้เป็น Features พื้นฐาน

In [ ]:
# คอลัมน์ที่ไม่นำมาใช้เป็น Feature
drop_cols_base = ["Duration(min)", "Distance(mi)", "Start_Time", "Description", "Street", 
                  "City", "County", "Zipcode", "Start_Date", "State", "Sunrise_Sunset",
                  "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]

base_features = [col for col in df_train.columns if col not in drop_cols_base]

X_base = df_train[base_features]

# แยกประเภทคอลัมน์
num_features = X_base.select_dtypes(include=['float32', 'float64', 'int32', 'int64', 'uint8']).columns.tolist()
cat_features = X_base.select_dtypes(include=['object']).columns.tolist()

print(f"Base Features: {len(base_features)}")
print(f"- Numerical: {len(num_features)}")
print(f"- Categorical: {len(cat_features)}")

## 3. Model 1: Distance Predictor
เป้าหมาย: ทำนาย `Distance(mi)`

In [ ]:
y_distance = df_train["Distance(mi)"]

# Preprocessing
preprocessor_dist = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ]
)

model_dist = Pipeline(steps=[
    ('preprocessor', preprocessor_dist),
    ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, tree_method='hist'))
])

print("Training Model 1 (Distance)...")
# ใช้ 20% ของข้อมูลเพื่อความรวดเร็วในการรันครั้งแรก
X_train_d, X_val_d, y_train_d, y_val_d = train_test_split(X_base, y_distance, test_size=0.2, random_state=42)
model_dist.fit(X_train_d, y_train_d)

dist_mae = mean_absolute_error(y_val_d, model_dist.predict(X_val_d))
print(f"Model 1 MAE: {dist_mae:.4f}")

## 4. Model 2: Duration Predictor
เป้าหมาย: ทำนาย `Duration(min)` โดยใช้ `Distance(mi)` เข้ามาเสริม

In [ ]:
y_duration = df_train["Duration(min)"]

# เพิ่ม Distance(mi) เข้าไปใน Features สำหรับ Model 2
X_duration = X_base.copy()
X_duration["Distance(mi)"] = df_train["Distance(mi)"] # ใช้ Ground Truth ในการ Train

# อัปเดต Numerical Features
num_features_dur = num_features + ["Distance(mi)"]

preprocessor_dur = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features_dur),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ]
)

model_dur = Pipeline(steps=[
    ('preprocessor', preprocessor_dur),
    ('regressor', XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, tree_method='hist'))
])

print("Training Model 2 (Duration)...")
X_train_t, X_val_t, y_train_t, y_val_t = train_test_split(X_duration, y_duration, test_size=0.2, random_state=42)
model_dur.fit(X_train_t, y_train_t)

dur_mae = mean_absolute_error(y_val_t, model_dur.predict(X_val_t))
print(f"Model 2 MAE: {dur_mae:.4f}")

## 5. Save Models

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model_dist, "../models/chain_dist_model.pkl")
joblib.dump(model_dur, "../models/chain_dur_model.pkl")

print("Models saved to models/ folder")